# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (do NOT subscript or iterate over dataset.metadata)
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review all available record sets and their corresponding field `@id`s. For clarity, all references are made using the `@id` attribute, per the Croissant schema.

In [ ]:
# List all record sets and their associated fields by @id
print("List of available record sets and their fields (@id):\n")
record_sets = []  # to use in later code for extraction
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    field_ids = []
    for field in record_set.fields:
        print(f"    - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")
        field_ids.append(field.id)
    print()
    record_sets.append(record_set.id)

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. We use the record set and field `@id`s from the previous section for explicit referencing.

The `dataframes` dictionary will store each DataFrame using its record set `@id` as key.

In [ ]:
# Load each record set into a DataFrame using its @id
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
    print(f" - Columns (@id): {df.columns.tolist()}")
    print(f" - Number of rows: {len(df)}\n")

# Preview the first record set (if any)
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Preview of data from RecordSet @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll perform basic EDA steps:
- Filtering on a numeric field (by `@id`) from a chosen record set
- Normalizing that field
- Grouping by a categorical attribute, if available

First, select a main numeric field and optionally a group field using the fields listed previously (by `@id`).

In [ ]:
# --- EDA on the primary RecordSet ---
# Pick the main record set (already assigned as main_record_set_id)

df = dataframes[main_record_set_id]

# Find a numeric field to work with (e.g., 'cr:coverage' or any MW, abundance field by @id)
# We'll attempt to automatically pick a float/integer field from the schema
numeric_field_id, group_field_id = None, None
main_record_set = None
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        main_record_set = rs
        for f in rs.fields:
            # Heuristic: look for Float or Integer types
            if (f.data_type is not None) and (f.data_type.lower() in ['float', 'double', 'integer', 'number']):
                numeric_field_id = f.id
                break
        # Try to find a categorical field for grouping (often 'cr:description', 'cr:accession', etc.)
        for f in rs.fields:
            if (f.data_type is not None) and (f.data_type.lower() == 'text'):
                group_field_id = f.id
                break
        break

# Print selected fields
print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}\n")

# Proceed only if a numeric field is found
if numeric_field_id is not None and numeric_field_id in df.columns:
    # Try to convert to numeric, with coercion for possible text entries
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() is not None else 0
    print(f"Applying filter: {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records (count): {len(filtered_df)}\n")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found suitable for EDA in this record set.")

## 5. Visualization
Below, we visualize the distribution of the chosen numeric field using a histogram and, if possible, display a mean aggregation by category if a group field was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        # Show boxplot by group
        plt.figure(figsize=(12,4))
        # Avoid too many categories
        if df[group_field_id].nunique() < 30:
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.xticks(rotation=45)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion
In this notebook, we've leveraged the `mlcroissant` library to load and analyze the dataset described by the Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). We've:

- Listed all record sets and their structure using their `@id`
- Loaded main data tables into Pandas
- Performed simple filtering, normalization, and grouping operations on numeric attributes
- Visualized field distributions

All references to fields, columns, and record sets are by their unique `@id` values, ensuring traceable and reproducible analyses.

**Next steps** might include advanced analysis, export, or machine learning using the now-structured DataFrames.